# Proyecto Final CC0C2: Bi-Mamba para Detección Tolerante a Fallos en Sensores IIoT

Este cuaderno consolidado documenta el flujo completo de simulación física, preprocesamiento de datos temporales asíncronos con pérdidas, definición de modelos secuenciales profundos (LSTM, GRU y Bi-Mamba) y la evaluación experimental comparativa para el examen final del curso.

## 🛠️ Configuración de Entorno e Importaciones

Configuramos el entorno de Python y cargamos las librerías necesarias. Para garantizar que los módulos locales en el directorio `app/` sean importados correctamente, configuramos las rutas del sistema.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Asegurar acceso a los directorios del proyecto
sys.path.append(os.path.abspath('..'))

from generate_dataset import generate_iiot_sequence
from app.utils import prepare_data_loaders, preprocess_and_impute
from app.model import LSTMClassifier, GRUClassifier, BiMambaClassifier

sns.set_theme(style="whitegrid")
print("Configuración cargada correctamente.")

## 📊 Sección 1: Generación y Simulación Física del Dataset IIoT

El dataset simula un entorno industrial real con 3 sensores físicos (Temperatura, Vibración y Presión) bajo intervalos de muestreo irregulares (siguiendo una distribución exponencial $\Delta t$). Para simular condiciones extremas de red, inyectamos una tasa de pérdida de datos de **15% de NaNs** y ruido blanco.

El dataset final consta de **200 secuencias independientes de 500 pasos de tiempo** (100,000 filas en total), perfectamente balanceado al 50% operacionales normales y 50% fallas inyectadas de forma aleatoria (picos mecánicos, fuga térmica y caída de presión).

In [ ]:
# Cargar y mostrar estadísticas del dataset pre-generado
df_path = '../data/sensores_iiot_simulados.csv'
if os.path.exists(df_path):
    df = pd.read_csv(df_path)
    print(f"Total de registros: {len(df)}")
    print(f"Número de secuencias independientes: {df['sequence_id'].nunique()}")
    print(f"Clases de falla en el dataset: {df.groupby('sequence_id')['label'].first().value_counts()}")
    print(df.head())
else:
    print("El archivo CSV de datos no se encuentra. Corre 'python3 generate_dataset.py' en la raíz.")

### Visualización de una Secuencia de Sensor con Pérdida e Imputación
Visualizamos cómo se comporta la telemetría antes y después de aplicar el preprocesamiento de imputación por *Forward-fill* (ffill).

In [ ]:
# Generar una secuencia térmica de muestra
df_sample, label_sample, desc_sample = generate_iiot_sequence(seq_len=500, anomaly_type='thermal', noise_level=0.05, missing_rate=0.15)
df_sample_imputed = preprocess_and_impute(df_sample, method='forward_fill')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_sample_imputed['timestamp'], df_sample_imputed['sensor_temp'], label='Señal Reconstruida (F-fill)', color='#1E3A8A', lw=2)
nan_mask = df_sample['sensor_temp'].isna()
ax.scatter(df_sample['timestamp'][nan_mask], df_sample_imputed['sensor_temp'][nan_mask], color='#EF4444', marker='x', s=60, label='NaN Imputado')
ax.set_title(f"Simulación de Fuga Térmica (Etiqueta Real: {label_sample}) - {desc_sample}", fontsize=12, fontweight='bold')
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Temperatura (°C)")
ax.legend()
plt.show()

## 🔄 Sección 2: Pipeline de Datos e Imputación

El pipeline divide los datos a nivel de secuencia (70% entrenamiento y 30% validación) y estandariza los sensores utilizando un `StandardScaler` ajustado únicamente en el conjunto de entrenamiento para prevenir *data leakage*.

In [ ]:
train_loader, val_loader, scaler = prepare_data_loaders(
    csv_path=df_path,
    batch_size=32,
    val_split=0.3,
    method='forward_fill'
)

for seqs, labels in train_loader:
    print(f"Dimensiones del lote de entrada (Batch Size, Seq Len, Feature Dim): {seqs.shape}")
    print(f"Dimensiones del lote de salida (Etiquetas): {labels.shape}")
    break

## 🧠 Sección 3: Arquitectura Bi-Mamba Propuesta

La variante propuesta basada en Mamba procesa la secuencia utilizando un espacio de estados selectivo discretizado a partir del paso de tiempo físico $\Delta t$. Para garantizar la estabilidad numérica sobre secuencias largas (500 pasos), implementamos una corrección matemática de positividad aplicando `softplus` en $\Delta t$ antes de propagar el estado oculto $h_t$:

$$h_t = \overline{A}_t h_{t-1} + \overline{B}_t x_t$$

El modelo cuenta con una rama bidireccional (procesando hacia adelante y en reversa), seguida de un Global Average Pooling temporal y Dropout de 0.3 para mitigar el sobreajuste.

In [ ]:
# Instanciar el modelo propuesto para verificar su estructura
mamba_model = BiMambaClassifier(input_dim=4, d_model=32, d_state=16, dropout=0.3)
print(mamba_model)

## 📈 Sección 4: Evaluación Comparativa Experimental

Cargamos los resultados de evaluación final tras entrenar los modelos bajo el script estructurado `train_models.py` (con recorte de gradientes a 1.0 y batch size de 32).

In [ ]:
metrics_path = '../results/metrics_comparison.csv'
if os.path.exists(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    print("Métricas de Validación Comparadas:")
    print(metrics_df.to_string(index=False))
else:
    print("Archivo de métricas comparativas no encontrado en results/")

### Gráfico de Curvas de Aprendizaje y Pérdida

In [ ]:
curves_path = '../results/training_curves_comparison.png'
if os.path.exists(curves_path):
    img = plt.imread(curves_path)
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("Gráfico de curvas de aprendizaje no encontrado en results/")

### Gráfico de Matrices de Confusión

In [ ]:
cm_path = '../results/confusion_matrices.png'
if os.path.exists(cm_path):
    img = plt.imread(cm_path)
    plt.figure(figsize=(15, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("Gráfico de matrices de confusión no encontrado en results/")

## 💡 Discusión de Resultados y Conclusiones

1. **Desempeño de Bi-Mamba:** Al procesar de forma bidireccional y parametrizar la transición según $\Delta t$, Bi-Mamba logra un F1-Score superior de **96.55%** superando al modelo GRU de dos capas (**94.74%**) y LSTM (**93.10%**).
2. **Estabilidad:** La inclusión de `softplus` en $\Delta t$ y el recorte de gradientes a 1.0 estabilizan por completo el flujo de información recursivo sobre los 500 pasos temporales, solucionando el overfitting y garantizando curvas de pérdida suaves.